In [ ]:
import os, math, random

import numpy as np
import pandas as pd
import statsmodels.api as sm

from tqdm import tqdm
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from pathlib import Path

### Library

In [ ]:
""" Data generators/parsers """

def relative_noise(arr, scale):
    if scale == 0.0:
        return np.zeros_like(arr)
    
    return np.random.normal(loc=0.0, scale=np.abs(arr) * scale)

def generate_scenario_data(rows: int, scenario: str, miss_col: int, labels_noise_scale: float):
    # Generate data as in https://www.nature.com/articles/s41467-020-19270-2 scenarios
    assert scenario in {"linear", "logistic"}, f"Invalid scenario for MI data {scenario}"
    feature_count = 2

    # x_2 is from U(-3, 3)
    x_2 = np.random.rand(rows) * 6 - 3

    if scenario == "linear":
        # Linear x_1 is from normal(mu=0.2 - 0.5*x_2, sigma=1.0)
        x_1 = np.random.normal(loc=0.2 - 0.5 * x_2, scale=1.0)
    else:
        # Logistic x_1 is from bernoulli(n=1, p=1 / (1 + exp(-0.2 + 0.5*x_2)))
        x_1 = np.random.binomial(n=1, p=1 / (1 + np.exp(-0.2 + 0.5 * x_2)))
    
    non_tampered_data = np.hstack((np.expand_dims(x_1, axis=1), np.expand_dims(x_2, axis=1)))
    
    miss_rows = []
    tampered_data = non_tampered_data.copy()
    
    non_tampered_labels = non_tampered_data @ np.ones((feature_count, 1)) + 1.0  # Weights and bias are ones
    non_tampered_labels += relative_noise(non_tampered_labels, scale=labels_noise_scale)
    for i in range(rows):
        # The variable is missing with probability 1 / (1 + math.exp(-0.3 + 0.2 * labels[i][0] - 0.1 * x_2[i]))
        if (1 / (1 + math.exp(-0.3 + 0.2 * non_tampered_labels[i][0] - 0.1 * x_2[i]))) > random.random():
            tampered_data[i][miss_col] = np.nan
            miss_rows.append(i)

    print(f"Generated data missing rate: {len(miss_rows)}/{len(non_tampered_data)}")
    return tampered_data, non_tampered_labels, non_tampered_data, miss_rows

def parse_real_data(outlier_threshold):
    # Real data
    df = pd.read_csv("../../data/mi/real/gcasr.csv")
    binary_columns = ["RaceAA", "RaceW", "HlthInsM", "HlthInsN", "Day", "NPO", "MedHisST", "MedHisTI", "MedHisVP", "MedHisFHSTK"]
    continuous_columns = ["NIHStrkS", "EMSNote", "LipTotal", "Age", "Gender"]
    col_types = ["lin" if i < len(continuous_columns) else "log" for i in range(len(binary_columns) + len(continuous_columns))]
    df_of_interest = df[continuous_columns + binary_columns]

    # Scale binary columns from [0, 1] to [-1, 1]
    df_of_interest[binary_columns] = df_of_interest[binary_columns] * 2 - 1
    # Normalize continuous columns
    df_of_interest[continuous_columns] = (df_of_interest[continuous_columns] - df_of_interest[continuous_columns].mean()) / df_of_interest[continuous_columns].std()

    # Fill the missing data for numerical feasibility
    real_data = df_of_interest.to_numpy()
    # Drop the missing data to get the training data
    real_complete_data = df_of_interest.dropna()

    # Get labels
    x = df["x"].fillna(0)
    # Reduce the outliers
    x[x < -outlier_threshold] = -outlier_threshold
    x[x > outlier_threshold] = outlier_threshold
    # Normalize
    x = (x - x.mean()) / x.std()
    real_labels = x.to_numpy()

    # Get missing rows
    real_miss_rows = df_of_interest.isna().to_numpy().astype(int).T

    return real_data, real_complete_data.to_numpy(), real_labels, real_miss_rows, col_types

def parse_simice_data(rows, miss_rows, miss_col, labels_noise_scale):
    non_tampered_data = np.loadtxt(f"../../data/mi/siMICE/scenario_1/{rows}/non_tampered_data.txt")
    cols = non_tampered_data.shape[1]
    
    non_tampered_labels = non_tampered_data @ np.ones((cols, 1)) + 1.0  # Weights and bias are ones
    non_tampered_labels += relative_noise(non_tampered_labels, scale=labels_noise_scale)

    tampered_data = non_tampered_data.copy()
    for i in range(miss_rows):
        tampered_data[i][miss_col] = np.nan
    
    return tampered_data, non_tampered_labels, non_tampered_data, list(range(miss_rows))

def savetxt(path, data, fmt):
    dirname = os.path.split(path)[0]
    Path(dirname).mkdir(parents=True, exist_ok=True)
    np.savetxt(path, data, fmt)

def export_data(name, data, labels, complete_data, miss_rows, miss_val):
    Path(f"../../data/mi/{name}/").mkdir(parents=True, exist_ok=True)
    
    np.savetxt(f"../../data/mi/{name}/tampered_data.txt", np.nan_to_num(data, nan=miss_val), fmt='%10.2f')
    np.savetxt(f"../../data/mi/{name}/non_tampered_data.txt", complete_data, fmt='%10.2f')
    np.savetxt(f"../../data/mi/{name}/non_tampered_labels.txt", labels, fmt='%10.3f')
    np.savetxt(f"../../data/mi/{name}/miss_rows.txt", miss_rows, fmt='%d')

In [ ]:
""" Imputers """

class Imputer:
    def __init__(self, model):
        self.model = model
    
    @staticmethod
    def split_train_test(data, target_col: int, miss_rows):
        m, n = data.shape
        miss_count = len(miss_rows)

        complete_rows = np.zeros((m - miss_count, n - 1))
        complete_labels = np.zeros((m - miss_count,))
        incomplete_rows = np.zeros((miss_count, n - 1))

        x_idx = 0
        x_com_idx = 0
        for i in range(m):
            row = data[i]
            row_complement = np.delete(row, target_col)
            if i in miss_rows:
                incomplete_rows[x_com_idx] = row_complement
                x_com_idx += 1
            else:
                complete_rows[x_idx] = row_complement
                complete_labels[x_idx] = row[target_col]
                x_idx += 1
        
        return complete_rows, incomplete_rows, complete_labels
    
    def fit(self, complete_data, complete_labels):
        self.model.fit(X=complete_data, y=complete_labels)
    
    def impute(self, data, miss_rows, miss_col: int, noise: bool):
        complete_data, incomplete_data, complete_labels = Imputer.split_train_test(data, miss_col, miss_rows)
        
        if len(incomplete_data) == 0:
            return complete_data.copy()
        
        self.model.fit(X=complete_data, y=complete_labels)
        imputed_y = self.model.predict(incomplete_data)
        if noise:
            imputed_y += np.random.normal(0.0, 0.01, size=imputed_y.shape)

        idx = 0
        imputed_data = data.copy()
        for i in miss_rows:
            imputed_data[i, miss_col] = imputed_y.ravel()[idx]
            idx += 1

        return imputed_data
    
    def impute(self, data, incomplete_data, miss_rows, miss_col, noise):
        imputed_y = self.model.predict(incomplete_data)
        if noise:
            imputed_y += np.random.normal(0.0, 0.01, size=imputed_y.shape)

        idx = 0
        imputed_data = data.copy()
        for i in miss_rows:
            imputed_data[i, miss_col] = imputed_y.ravel()[idx]
            idx += 1

        return imputed_data
    
    def impute_inplace(self, data, mask, miss_col, noise):
        if mask.sum() == 0:  # Nothing to impute --- no missing data
            return
        
        reference_data = np.hstack((data[:, :miss_col], data[:, miss_col+1:]))
        imputed_y = self.model.predict(reference_data[np.where(mask == 1)])
        if noise and isinstance(self.model, LinearRegression):
            imputed_y += np.random.normal(0.0, 0.01, size=imputed_y.shape)

        idx = 0
        for i in range(len(mask)):
            if mask[i]:
                data[i, miss_col] = imputed_y.ravel()[idx]
                idx += 1
    

class MI:
    def __init__(self, factor: int, impute_model, fit_model):
        self.imputer = Imputer(impute_model)
        self.model = fit_model
        self.factor = factor

    @staticmethod
    def rubin(weights):
        return np.mean(weights, axis=0)
    
    def fit(self, data, labels, miss_rows, miss_col, noise):
        complete_data, incomplete_data, complete_labels = Imputer.split_train_test(data, miss_col, miss_rows)
        
        imputed_data = []
        self.imputer.fit(complete_data=complete_data, complete_labels=complete_labels)
        
        imputed_data = [self.imputer.impute(
            data=data, incomplete_data=incomplete_data,
            miss_rows=miss_rows, miss_col=miss_col, noise=noise) for _ in range(self.factor)]
        
        self.fit_analysis_model(imputed_data, labels)
        return self, imputed_data
    
    def fit_analysis_model(self, imputed_data, labels):
        weights = np.zeros((self.factor, imputed_data[0].shape[1]))
        for i, data in enumerate(imputed_data):
            self.model.fit(X=data, y=labels)
            weights[i] = self.model.coef_.copy()
        
        self.model.coef_ = MI.rubin(weights)
    

class BatchMI:
    def __init__(self, factor: int, impute_models, fit_model):
        self.imputers = [Imputer(impute_model) for impute_model in impute_models]
        self.model = fit_model
        self.factor = factor

    def fit(self, data, complete_data, labels, miss_rows):
        miss_col = 0
        for imputer in self.imputers:
            train_data = np.hstack((complete_data[:, :miss_col], complete_data[:, miss_col+1:]))
            train_labels = complete_data[:, miss_col:miss_col+1]
            
            imputer.fit(complete_data=train_data, complete_labels=train_labels.ravel())
            miss_col += 1
        
        imputed_data = []
        weights = np.zeros((self.factor, data.shape[1]))
        for i in range(self.factor):
            inter_imputed_data = data.copy()
            self.impute_inplace(inter_imputed_data, miss_rows)
            imputed_data.append(inter_imputed_data)
            self.model.fit(X=inter_imputed_data, y=labels)
            weights[i] = self.model.coef_.copy()
        
        self.model.coef_ = MI.rubin(weights)
        return self, imputed_data
    
    def impute_inplace(self, incomplete_data, miss_rows):
        miss_col = 0
        for imputer in self.imputers:
            imputer.impute_inplace(
                data=incomplete_data,
                mask=miss_rows[miss_col], miss_col=miss_col, noise=True)
            miss_col += 1


class ClosedFormReg:
    def fit(self, X, y):
        X_tilde = np.hstack((X, np.ones((X.shape[0], 1))))
        coef = np.linalg.inv(X_tilde.T @ X_tilde) @ X_tilde.T @ y
        self.coef_ = coef[:-1].flatten()
        self.intercept = coef[-1]

    def predict(self, X):
        return X @ np.expand_dims(self.coef_, axis=1) + self.intercept

In [ ]:
""" PyMICE """

def pymice(incomplete_data):
    imp_mean = IterativeImputer()  # set sample_posterior=True, random_state=random.randint(0, 1000) as attributes for MI
    imp_mean.fit(incomplete_data)
    return imp_mean.transform(incomplete_data)

In [ ]:
""" Evaluators """

def sign(x):
    return -1 if x < 0 else 1 if x > 0 else 0

def get_stat_sig(X, y):
    X2 = sm.add_constant(X)
    est = sm.OLS(np.expand_dims(y, axis=1), X2)
    est2 = est.fit()
    return est2.params, est2.pvalues

def discrepancies(data_instance, mdni_stats, sequre_stats, verbose=False):
    mdni_coeff, mdni_pval = mdni_stats
    sequre_coeff, sequre_pval = sequre_stats

    discr = 0
    for i in range(len(mdni_coeff)):
        if mdni_pval[i] <= 0.05 and (sequre_pval[i] > 0.05 or sign(mdni_coeff[i]) != sign(sequre_coeff[i])):
            if verbose:
                print(f"Discrepancy found at var {i} and instance {data_instance}! SIMICE p_val: {mdni_pval[i]}, Sequre p_val: {sequre_pval[i]}, SIMICE coeff: {mdni_coeff[i]}, Sequre coeff: {sequre_coeff[i]}")
            discr += 1
        elif verbose:
            print(f"No discrepancy found at var {i} and instance {data_instance}:\n\t\tSIMICE p_val: {mdni_pval[i]}\n\t\tSequre p_val: {sequre_pval[i]}\n\t\tSIMICE coeff: {mdni_coeff[i]}\n\t\tSequre coeff: {sequre_coeff[i]}")
    
    return discr

def count_discrepancies(imputed_datasets, mice_imputed_data, labels):
    mice_stats = get_stat_sig(mice_imputed_data, labels)

    discr = 0
    for i, imputed_dataset in enumerate(imputed_datasets):
        sequre_stats = get_stat_sig(imputed_dataset, labels)
        discr += discrepancies(i, mice_stats, sequre_stats)
    
    return discr / len(imputed_datasets)

def evaluate_scenario(msg, scenario, impute_model, mi_factor, miss_col, miss_val, dump_data):
    (tampered_data, labels, non_tampered_data, miss_rows) = scenario

    mice_imputed_data = pymice(tampered_data)
    if dump_data:
        savetxt(f"../../data/mi/{msg}/py_mice_data.txt", mice_imputed_data, fmt='%10.2f')
    
    fit_model = ClosedFormReg()
    mi, imputed_datasets = MI(mi_factor, impute_model, fit_model).fit(
        np.nan_to_num(tampered_data, nan=miss_val), labels, miss_rows, miss_col, noise=True if msg == "scenario_1" else False)

    predicted_data = mi.model.predict(non_tampered_data).flatten()
    labels = labels.flatten()
    assert predicted_data.shape == labels.shape

    abs_diff = np.abs(predicted_data - labels)
    y_hat_y_mean = np.mean(abs_diff)
    y_hat_y_std = np.std(abs_diff)
    discrepancies_count = count_discrepancies(imputed_datasets, mice_imputed_data, labels)

    return mi.model.coef_, y_hat_y_mean, y_hat_y_std, discrepancies_count

def benchmark_scenario(msg, runs_count, scenario, impute_model, mi_factor, miss_col, miss_val, dump_data):
    features_count = scenario[0].shape[1]
    
    coefs = np.zeros((runs_count, features_count))
    y_hat_y_means = np.zeros((runs_count,))
    y_hat_y_sds = np.zeros((runs_count,))
    discrs = np.zeros((runs_count,))
    
    for i in tqdm(range(runs_count), "Evaluating scenario"):
        coef, y_hat_y_mean, y_hat_y_sd, discr = evaluate_scenario(
            msg, scenario, impute_model, mi_factor, miss_col, miss_val, dump_data)
        coefs[i] = coef
        y_hat_y_means[i] = y_hat_y_mean
        y_hat_y_sds[i] = y_hat_y_sd
        discrs[i] = discr
    
    return coefs, y_hat_y_means, y_hat_y_sds, discrs

def evaluate_real_setup(real_data, mi_factor, miss_val, dump_data):
    incomplete_data, complete_data, labels, miss_rows, col_types = real_data

    impute_models = [LogisticRegression() if t == "log" else LinearRegression() for t in col_types]
    fit_model = LinearRegression()
    
    mice_imputed_data = pymice(incomplete_data)
    if dump_data:
        savetxt(f"../../data/mi/real/py_mice_data.txt", mice_imputed_data, fmt='%10.2f')

    incompletes_w_miss_val = np.nan_to_num(incomplete_data, nan=miss_val)
    mi, imputed_datasets = BatchMI(mi_factor, impute_models, fit_model).fit(
        incompletes_w_miss_val, complete_data, labels, miss_rows)
    
    predicted_data = mi.model.predict(incompletes_w_miss_val).flatten()
    labels = labels.flatten()
    assert predicted_data.shape == labels.shape

    abs_diff = np.abs(predicted_data - labels)
    y_hat_y_mean = np.mean(abs_diff)
    y_hat_y_sd = np.std(abs_diff)
    discrepancies_count = count_discrepancies(imputed_datasets, mice_imputed_data, labels)

    return mi.model.coef_, y_hat_y_mean, y_hat_y_sd, discrepancies_count

def benchmark_real_setup(runs_count, real_data, mi_factor, miss_val, dump_data):
    y_hat_y_means = np.zeros((runs_count,))
    y_hat_y_sds = np.zeros((runs_count,))
    discrs = np.zeros((runs_count,))
    
    for i in tqdm(range(runs_count), "Evaluating real setup"):
        _, y_hat_y_mean, y_hat_y_sd, discr = evaluate_real_setup(real_data, mi_factor, miss_val, dump_data)
        y_hat_y_means[i] = y_hat_y_mean
        y_hat_y_sds[i] = y_hat_y_sd
        discrs[i] = discr
    
    return None, y_hat_y_means, y_hat_y_sds, discrs
    
def evaluate_pymice(data, labels, non_tampered_data, mi_factor):
    fit_model = LinearRegression()
    imputed_datasets = [pymice(data) for _ in range(mi_factor)]

    mi = MI(mi_factor, None, fit_model)
    mi.fit_analysis_model(imputed_datasets, labels)
    
    predicted_data = mi.model.predict(non_tampered_data).flatten()
    labels = labels.flatten()
    assert predicted_data.shape == labels.shape

    abs_diff = np.abs(predicted_data - labels)
    y_hat_y_mean = np.mean(abs_diff)
    y_hat_y_sds = np.std(abs_diff)

    return mi.model.coef_, y_hat_y_mean, y_hat_y_sds

def benchmark_pymice(runs_count, data, labels, non_tampered_data, mi_factor):
    coefs = np.zeros((runs_count, data.shape[1]))
    y_hat_y_means = np.zeros((runs_count,))
    y_hat_y_sds = np.zeros((runs_count,))
    discrs = np.zeros((runs_count,)) - 1
    
    for i in tqdm(range(runs_count), "Evaluating pymice"):
        coef, y_hat_y_mean, y_hat_y_sd = evaluate_pymice(data, labels, non_tampered_data, mi_factor)
        
        coefs[i] = coef
        y_hat_y_means[i] = y_hat_y_mean
        y_hat_y_sds[i] = y_hat_y_sd
    
    return coefs, y_hat_y_means, y_hat_y_sds, discrs

def evaluate_simice(scenario, tampered_data, labels, non_tampered_data, mi_factor, dump_data):
    rows = tampered_data.shape[0]
    
    fit_model = LinearRegression()
    imputed_datasets = [np.loadtxt(f"../../data/mi/siMICE/{scenario}/{rows}/imputed_data_{i + 1}.txt") for i in range(mi_factor)]
    mice_imputed_data = pymice(tampered_data)
    if dump_data:
        savetxt(f"../../data/mi/siMICE/{scenario}/{rows}/py_mice_data.txt", mice_imputed_data, fmt='%10.2f')
    
    mi = MI(mi_factor, None, fit_model)
    mi.fit_analysis_model(imputed_datasets, labels)
    
    predicted_data = mi.model.predict(non_tampered_data).flatten()
    labels = labels.flatten()
    assert predicted_data.shape == labels.shape
    
    abs_diff = np.abs(predicted_data - labels)
    y_hat_y_mean = np.mean(abs_diff)
    y_hat_y_sds = np.std(abs_diff)
    discrepancies_count = count_discrepancies(imputed_datasets, mice_imputed_data, labels)

    return mi.model.coef_, y_hat_y_mean, y_hat_y_sds, discrepancies_count

def benchmark_simice(runs_count, scenario, tampered_data, labels, non_tampered_data, mi_factor, dump_data):
    cols = tampered_data.shape[1]
    coefs = np.zeros((runs_count, cols))
    y_hat_y_means = np.zeros((runs_count,))
    y_hat_y_sds = np.zeros((runs_count,))
    discrs = np.zeros((runs_count,))
    
    for i in tqdm(range(runs_count), "Evaluating siMICE"):
        coef, y_hat_y_mean, y_hat_y_sd, discr = evaluate_simice(scenario, tampered_data, labels, non_tampered_data, mi_factor, dump_data)
        coefs[i] = coef
        y_hat_y_means[i] = y_hat_y_mean
        y_hat_y_sds[i] = y_hat_y_sd
        discrs[i] = discr
    
    return coefs, y_hat_y_means, y_hat_y_sds, discrs

def print_stats(msg, coefs, biases, sds, discrepancies_count):
    print(f"{msg} | Python MI y_hat - y mean | mean: {np.mean(biases)}, min: {np.min(biases)}, max: {np.max(biases)}")
    print(f"{msg} | Python MI y_hat - y SD | mean: {np.mean(sds)}, min: {np.min(sds)}, max: {np.max(sds)}")
    print(f"{msg} | Python MI # of discrepancies w.r.t. PyMICE per imputed dataset | mean: {np.mean(discrepancies_count)}, min: {np.min(discrepancies_count)}, max: {np.max(discrepancies_count)}")
    
    if msg.lower().startswith("scenario") or msg.lower().startswith("pymice") or msg.lower().startswith("simice"):
        coefs_truth = np.ones_like(coefs[0])
        coefs_bias = np.linalg.norm(np.mean(coefs, axis=0) - coefs_truth)
        coefs_sd = np.sqrt(np.mean(np.sum((coefs - np.mean(coefs, axis=0)) ** 2, axis=1)))
        coefs_rmse = np.sqrt(np.mean(np.sum((coefs - coefs_truth) ** 2, axis=1)))

        print(f"{msg} | Python MI coefs bias: {coefs_bias}")
        print(f"{msg} | Python MI coefs SD: {coefs_sd}")
        print(f"{msg} | Python MI coefs rMSE: {coefs_rmse}")

### Setup

In [ ]:
run_scenarios = True
run_pymice = False
run_simice = False
run_real_setup = False

scenarios_runs_count = 1000
pymice_runs_count = 10
simice_runs_count = 1
real_setup_runs_count = 5

mi_factor = 5
miss_val = 0.0
dump_data = True
labels_noise_scale = 0.01

scenarios_rows = 500
scenarios_miss_col = 0

real_setup_outlier_threshold = 3000.0

simice_rows = 500
simice_miss_rows = 150

random.seed(0)
np.random.seed(0)

### Data prep

In [ ]:
# Data generated by siMICE scripts (https://github.com/Luyaochen1/midn_gui)
simice_sc1 = parse_simice_data(simice_rows, simice_miss_rows, scenarios_miss_col, labels_noise_scale)

In [ ]:
# Scenario 1
sc1 = generate_scenario_data(scenarios_rows, "linear", scenarios_miss_col, labels_noise_scale)

In [ ]:
# Scenario 2
sc2 = generate_scenario_data(scenarios_rows, "logistic", scenarios_miss_col, labels_noise_scale)

In [ ]:
if dump_data:
    export_data("scenario_1", *sc1, miss_val)
    export_data("scenario_2", *sc2, miss_val)
    export_data("scenario_siMICE_1", *simice_sc1, miss_val)

### Imputation via regression analysis

In [ ]:
if run_scenarios:
    print_stats("Scenario siMICE", *benchmark_scenario("scenario_siMICE", scenarios_runs_count, simice_sc1, ClosedFormReg(), mi_factor, scenarios_miss_col, miss_val, dump_data))
    print_stats("Scenario 1", *benchmark_scenario("scenario_1", scenarios_runs_count, sc1, ClosedFormReg(), mi_factor, scenarios_miss_col, miss_val, dump_data))
    print_stats("Scenario 2", *benchmark_scenario("scenario_2", scenarios_runs_count, sc2, LogisticRegression(), mi_factor, scenarios_miss_col, miss_val, dump_data))

### PyMICE

In [ ]:
if run_pymice:
    print_stats("PyMICE siMICE scenario 1", *benchmark_pymice(pymice_runs_count, simice_sc1[0], simice_sc1[1], simice_sc1[2], mi_factor))
    print_stats("PyMICE scenario 1", *benchmark_pymice(pymice_runs_count, sc1[0], sc1[1], sc1[2], mi_factor))
    print_stats("PyMICE scenario 2", *benchmark_pymice(pymice_runs_count, sc2[0], sc2[1], sc2[2], mi_factor))

### siMICE

In [ ]:
if run_simice:
    print_stats("siMICE scenario 1", *benchmark_simice(simice_runs_count, "scenario_1", simice_sc1[0], simice_sc1[1], simice_sc1[2], mi_factor, dump_data))

### Real data

In [ ]:
if run_real_setup:
    real = parse_real_data(real_setup_outlier_threshold)
    print_stats("Real data", *benchmark_real_setup(real_setup_runs_count, real, mi_factor, miss_val, dump_data))
    print_stats("PyMICE real setup", *benchmark_pymice(real_setup_runs_count, real[0], real[2], mi_factor))